# <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;  " > Practice lab: Deep Learning cho Content-Based Filtering

Trong exercise này, bạn sẽ triển khai content-based filtering bằng cách sử dụng neural network để xây dựng recommender system cho phim. 

# Outline <img align="left" src="./images/film_reel.png"     style=" width:40px;  " >
- [ 1 - Packages](#1)
- [ 2 - Movie ratings dataset](#2)
  - [ 2.1 Content-based filtering with a neural network](#2.1)
  - [ 2.2 Preparing the training data](#2.2)
- [ 3 - Neural Network for content-based filtering](#3)
  - [ 3.1 Predictions](#3.1)
    - [ Exercise 1](#ex01)
- [ 4 - Congratulations!](#4)


<a name="1"></a>
## 1 - Packages <img align="left" src="./images/movie_camera.png"     style=" width:40px;  ">
Chúng ta sẽ sử dụng packages, NumPy, TensorFlow quen thuộc và các quy trình hữu ích từ [scikit-learn](https://scikit-learn.org/stable/). Chúng ta cũng sẽ sử dụng [tabulate](https://pypi.org/project/tabulate/) để in bảng một cách gọn gàng và [Pandas](https://pandas.pydata.org/) để sắp xếp dữ liệu dạng bảng.


In [5]:
import numpy as np
import numpy.ma as ma
from numpy import genfromtxt
from collections import defaultdict
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
import tabulate
from recsysNN_utils import *
pd.set_option("display.precision", 1)

<a name="2"></a>
## 2 - Xếp hạng phim dataset <img align="left" src="./images/film_rating.png" style=" width:40px;" >
Tập dữ liệu được lấy từ [MovieLens ml-latest-small](https://grouplens.org/datasets/movielens/latest/) dataset. 

[F. Maxwell Harper và Joseph A. Konstan. 2015. MovieLens Datasets: Lịch sử và bối cảnh. Giao dịch ACM trên Hệ thống thông minh tương tác (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

dataset ban đầu có 9000 phim được 600 user đánh giá với xếp hạng theo thang điểm từ 0,5 đến 5 với khoảng tăng 0,5 bước. dataset đã được giảm kích thước để tập trung vào các bộ phim từ những năm 2000 và các thể loại phổ biến. dataset rút gọn có user $n_u = 395$ và phim $n_m= 694$. Đối với mỗi bộ phim, dataset cung cấp tên phim, ngày phát hành và một hoặc nhiều thể loại. Ví dụ: "Toy Story 3" được phát hành vào năm 2010 và có một số thể loại: "Adventure|Animation|Children|Comedy|Fantasy|IMAX".  dataset này chứa ít thông tin về user ngoài xếp hạng của họ. dataset này được sử dụng để tạo training vectors cho các neural network được mô tả bên dưới.


<a name="2.1"></a>
### 2.1 Content-based filtering với neural network

Trong collaborative filtering lab, bạn đã tạo hai vectors, một user vector và một item/movie vector có dot product sẽ dự đoán xếp hạng. vectors chỉ được lấy từ xếp hạng.   

Content-based filtering cũng tạo user và movie feature vector nhưng nhận ra rằng có thể có thông tin khác về user và/hoặc phim có thể cải thiện prediction. Thông tin bổ sung được cung cấp cho neural network, sau đó tạo ra user và movie vector như hiển thị bên dưới.
<figure>
    <center> <img src="./images/RecSysNN.png"   style="width:500px;height:280px;" ></center>
</figure>
Nội dung phim được cung cấp cho network là sự kết hợp giữa dữ liệu gốc và một số 'features được thiết kế'. Nhớ rằng cuộc thảo luận về kỹ thuật feature và lab từ Khóa 1, Tuần 2, lab 4. features ban đầu là năm bộ phim được phát hành và thể loại phim được trình bày dưới dạng vector hấp dẫn. Có 14 thể loại. feature được thiết kế là xếp hạng trung bình được lấy từ xếp hạng của user. Phim có nhiều thể loại có training vector cho mỗi thể loại. 

Nội dung user chỉ bao gồm features được thiết kế. Xếp hạng trung bình cho mỗi thể loại được tính cho mỗi user. Ngoài ra, user id, số lượng xếp hạng và xếp hạng trung bình đều có sẵn nhưng không được bao gồm trong nội dung training hoặc prediction. Chúng rất hữu ích trong việc giải thích dữ liệu.

training set bao gồm tất cả các xếp hạng do user thực hiện trong tập dữ liệu. User và movie/item vectors được đưa vào network trên cùng nhau dưới dạng training set. User vector giống nhau đối với tất cả các phim được user đánh giá. 

Dưới đây, hãy tải và hiển thị một số dữ liệu.


In [2]:
# Tải dữ liệu, đặt biến cấu hình
item_train, user_train, y_train, item_features, user_features, item_vecs, movie_dict, user_to_genre = load_data()

num_user_features = user_train.shape[1] - 3  # remove userid, rating count and ave rating during training
num_item_features = item_train.shape[1] - 1  # remove movie id at train time
uvs = 3  # user genre vector start
ivs = 3  # item genre vector start
u_s = 3  # start of columns to use in training, user
i_s = 1  # start of columns to use in training, items
scaledata = True  # applies the standard scalar to data if true
print(f"Number of training vectors: {len(item_train)}")

Number of training vectors: 58187


Một số user và item/phim features không được sử dụng trong training. Bên dưới, features trong ngoặc "[]" chẳng hạn như "user id", "số xếp hạng" và "trung bình xếp hạng" không được bao gồm khi model được training và sử dụng. Lưu ý, user vector giống nhau đối với tất cả các phim được xếp hạng.


In [3]:
pprint_train(user_train, user_features, uvs,  u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9
2,16,4.1,3.9,5.0,0.0,0.0,4.0,4.2,4.0,4.0,0.0,3.0,4.0,0.0,4.2,3.9


In [4]:
pprint_train(item_train, item_features, ivs, i_s, maxcount=5, user=False)

[movie id],year,ave rating,Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
6874,2003,4.0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
6874,2003,4.0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
8798,2004,3.8,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8798,2004,3.8,0,0,0,0,0,1,0,0,0,0,0,0,0,0


In [5]:
print(f"y_train[:5]: {y_train[:5]}")

y_train[:5]: [4.  4.  4.  3.5 3.5]


Ở trên, chúng ta có thể thấy phim 6874 là một bộ phim hành động được phát hành năm 2003. User 2 đánh giá phim hành động trung bình là 3,9. Hơn nữa, phim 6874 cũng được xếp vào thể loại Crime và Horror. User MovieLens đã xếp hạng trung bình cho bộ phim là 4. Training example bao gồm một hàng từ cả hai bảng và xếp hạng từ y_train.


<a name="2.2"></a>
### 2.2 Chuẩn bị dữ liệu training
Nhớ rằng trong Khóa 1, Tuần 2, bạn đã khám phá feature scaling như một phương tiện để cải thiện khả năng hội tụ. Chúng ta sẽ mở rộng input features bằng cách sử dụng [scikit learn StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html). Điều này đã được sử dụng trong Khóa 1, Tuần 2, Lab 5. Dưới đây, inverse_transform cũng được hiển thị để tạo ra các đầu vào ban đầu.


In [6]:
# chia tỷ lệ dữ liệu training
if scaledata:
    item_train_save = item_train
    user_train_save = user_train

    scalerItem = StandardScaler()
    scalerItem.fit(item_train)
    item_train = scalerItem.transform(item_train)

    scalerUser = StandardScaler()
    scalerUser.fit(user_train)
    user_train = scalerUser.transform(user_train)

    print(np.allclose(item_train_save, scalerItem.inverse_transform(item_train)))
    print(np.allclose(user_train_save, scalerUser.inverse_transform(user_train)))

True
True


Để cho phép chúng ta đánh giá kết quả, chúng ta sẽ chia dữ liệu thành training và các bộ kiểm tra như đã thảo luận trong Khóa 2, Tuần 3. Ở đây chúng ta sẽ sử dụng [sklean train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) để phân tách và xáo trộn dữ liệu. Lưu ý rằng việc đặt trạng thái ngẫu nhiên ban đầu thành cùng một giá trị sẽ đảm bảo mục, user và y được xáo trộn giống hệt nhau.


In [7]:
item_train, item_test = train_test_split(item_train, train_size=0.80, shuffle=True, random_state=1)
user_train, user_test = train_test_split(user_train, train_size=0.80, shuffle=True, random_state=1)
y_train, y_test       = train_test_split(y_train,    train_size=0.80, shuffle=True, random_state=1)
print(f"movie/item training data shape: {item_train.shape}")
print(f"movie/item test  data shape: {item_test.shape}")

movie/item training data shape: (46549, 17)
movie/item test  data shape: (11638, 17)


Dữ liệu được chia tỷ lệ, xáo trộn hiện có giá trị trung bình bằng 0.


In [8]:
pprint_train(user_train, user_features, uvs, u_s, maxcount=5)

[user id],[rating count],[rating ave],Act ion,Adve nture,Anim ation,Chil dren,Com edy,Crime,Docum entary,Drama,Fan tasy,Hor ror,Mys tery,Rom ance,Sci -Fi,Thri ller
1,0,0.6,0.7,0.6,0.6,0.7,0.7,0.5,0.7,0.2,0.3,0.3,0.5,0.5,0.8,0.5
0,0,1.6,1.5,1.7,0.9,1.0,1.4,0.8,-1.2,1.2,1.2,1.6,0.9,1.4,1.2,1.0
0,0,0.8,0.6,0.7,0.5,0.6,0.6,0.3,-1.2,0.7,0.8,0.9,0.6,0.2,0.6,0.6
1,0,-0.1,0.2,-0.1,0.3,0.7,0.3,0.2,1.0,-0.5,-0.7,-2.1,0.5,0.7,0.3,0.0
-1,0,-1.3,-0.8,-0.8,0.1,-0.1,-1.1,-0.9,-1.2,-1.5,-0.6,-0.5,-0.6,-0.9,-0.4,-0.9


Chia tỷ lệ xếp hạng target bằng cách sử dụng Bộ chia tỷ lệ tối thiểu tối thiểu để chia tỷ lệ target nằm trong khoảng từ -1 đến 1. Chúng ta sử dụng scikit-learn vì nó có inverse_transform. [scikit learn MinMaxScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html)


In [9]:
scaler = MinMaxScaler((-1, 1))
scaler.fit(y_train.reshape(-1, 1))
ynorm_train = scaler.transform(y_train.reshape(-1, 1))
ynorm_test = scaler.transform(y_test.reshape(-1, 1))
print(ynorm_train.shape, ynorm_test.shape)

(46549, 1) (11638, 1)


<a name="3"></a>
## 3 - Neural Network cho content-based filtering
Bây giờ, hãy xây dựng neural network như được mô tả trong hình trên. Nó sẽ có hai network được kết hợp bởi dot product. Bạn sẽ xây dựng hai network. Trong ví dụ này, chúng sẽ giống hệt nhau. Lưu ý rằng các network này không cần phải giống nhau. Nếu nội dung user lớn hơn đáng kể so với nội dung phim, bạn có thể chọn tăng độ phức tạp của user network so với movie network. Trong trường hợp này, nội dung tương tự nhau nên network giống nhau.

- Sử dụng Keras sequential model
    - layer đầu tiên là dense layer với 256 units và relu activation.
    - layer thứ hai là dense layer với 128 units và relu activation.
    - layer thứ ba là dense layer với các unit `num_outputs` và activation tuyến tính hoặc không có.   
    
Phần còn lại của network sẽ được cung cấp. Code được cung cấp không sử dụng Keras sequential model mà thay vào đó sử dụng Keras [functional api](https://keras.io/guides/functional_api/). Định dạng này cho phép linh hoạt hơn trong cách các thành phần được kết nối với nhau.


In [10]:
# GRADED_CELL
# UNQ_C1

num_outputs = 32
tf.random.set_seed(1)
user_NN = tf.keras.models.Sequential([
    # ## START CODE HERE ###
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs, activation='linear'),
    # ## END CODE HERE ###
])

item_NN = tf.keras.models.Sequential([
    # ## START CODE HERE ###
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(num_outputs, activation='linear'),
    # ## END CODE HERE ###
])

# tạo người dùng input và trỏ đến mạng cơ sở
input_user = tf.keras.layers.Input(shape=(num_user_features))
vu = user_NN(input_user)
vu = tf.linalg.l2_normalize(vu, axis=1)

# tạo mục input và trỏ đến mạng cơ sở
input_item = tf.keras.layers.Input(shape=(num_item_features))
vm = item_NN(input_item)
vm = tf.linalg.l2_normalize(vm, axis=1)

# tính dot product của hai vectors vu và vm
output = tf.keras.layers.Dot(axes=1)([vu, vm])

# chỉ định đầu vào và đầu ra của model
model = Model([input_user, input_item], output)

model.summary()

Model: "model"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 14)]         0                                            
__________________________________________________________________________________________________
input_2 (InputLayer)            [(None, 16)]         0                                            
__________________________________________________________________________________________________
sequential (Sequential)         (None, 32)           40864       input_1[0][0]                    
__________________________________________________________________________________________________
sequential_1 (Sequential)       (None, 32)           41376       input_2[0][0]                    
______________________________________________________________________________________________

In [11]:
# Kiểm tra công khai
from public_tests import *
test_tower(user_NN)
test_tower(item_NN)

All tests passed!
All tests passed!


<details>
  <summary><font size="3" color="darkgreen"><b>Click để biết gợi ý</b></font></summary>
    
  Bạn có thể tạo dense layer bằng relu activation như minh họa.
    
```python     
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),

    
    ### END CODE HERE ###  
])
```    
<details>
    <summary><font size="2" color="darkblue"><b> Nhấp để tìm giải pháp</b></font></summary>
    
```python 
user_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])

item_NN = tf.keras.models.Sequential([
    ### START CODE HERE ###     
  tf.keras.layers.Dense(256, activation='relu'),
  tf.keras.layers.Dense(128, activation='relu'),
  tf.keras.layers.Dense(num_outputs),
    ### END CODE HERE ###  
])
```
</details>
</details>


Chúng ta sẽ sử dụng mean squared error loss và Adam optimizer.


In [12]:
tf.random.set_seed(1)
cost_fn = tf.keras.losses.MeanSquaredError()
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,
              loss=cost_fn)

In [13]:
tf.random.set_seed(1)
model.fit([user_train[:, u_s:], item_train[:, i_s:]], ynorm_train, epochs=30)

Train on 46549 samples
Epoch 1/30
46549/46549 [==============================] - 6s 133us/sample - loss: 0.1254
Epoch 2/30
46549/46549 [==============================] - 6s 122us/sample - loss: 0.1187
Epoch 3/30
46549/46549 [==============================] - 6s 121us/sample - loss: 0.1169
Epoch 4/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1154
Epoch 5/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1142
Epoch 6/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1130
Epoch 7/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1119
Epoch 8/30
46549/46549 [==============================] - 6s 119us/sample - loss: 0.1110
Epoch 9/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1095
Epoch 10/30
46549/46549 [==============================] - 6s 120us/sample - loss: 0.1083
Epoch 11/30
46549/46549 [==============================] - 6s 122us/sample - loss: 0.1

Đánh giá model để xác định loss trên dữ liệu test. Nó có thể so sánh với training loss, cho thấy model về cơ bản không quá khớp với dữ liệu training.


In [14]:
model.evaluate([user_test[:, u_s:], item_test[:, i_s:]], ynorm_test)

11638/11638 [==============================] - 0s 36us/sample - loss: 0.1045


0.10449595100221243

<a name="3.1"></a>
### 3.1 Predictions
Dưới đây, bạn sẽ sử dụng model của chúng ta để tạo predictions trong một số trường hợp. 
#### Predictions cho user mới
Đầu tiên, chúng ta sẽ tạo một user mới và model đề xuất phim cho user đó. Sau khi bạn đã thử ví dụ này trên nội dung user mẫu, vui lòng thay đổi nội dung user để phù hợp với sở thích của riêng bạn và xem model gợi ý những gì. Lưu ý rằng xếp hạng nằm trong khoảng từ 0,5 đến 5,0, tính theo gia số nửa bước.


In [15]:
new_user_id = 5000
new_rating_ave = 1.0
new_action = 1.0
new_adventure = 1
new_animation = 1
new_childrens = 1
new_comedy = 5
new_crime = 1
new_documentary = 1
new_drama = 1
new_fantasy = 1
new_horror = 1
new_mystery = 1
new_romance = 5
new_scifi = 5
new_thriller = 1
new_rating_count = 3

user_vec = np.array([[new_user_id, new_rating_count, new_rating_ave,
                      new_action, new_adventure, new_animation, new_childrens,
                      new_comedy, new_crime, new_documentary,
                      new_drama, new_fantasy, new_horror, new_mystery,
                      new_romance, new_scifi, new_thriller]])

Hãy cùng xem những bộ phim được xếp hạng cao nhất dành cho user mới. Nhớ rằng, user vector có thể loại ưa thích Comedy và Romance.
Dưới đây, chúng ta sẽ sử dụng bộ movie/item vectors, `item_vecs` có vector cho mỗi phim trong training/test set. Điều này phù hợp với user vector ở trên và vectors được chia tỷ lệ được sử dụng để dự đoán xếp hạng cho tất cả các phim cho user mới của chúng ta ở trên.


In [16]:
# tạo và sao chép người dùng vector để khớp với số lượng phim trong tập dữ liệu.
user_vecs = gen_user_vecs(user_vec,len(item_vecs))

# chia tỷ lệ vectors và tạo predictions cho tất cả các phim. Trả về kết quả được sắp xếp theo đánh giá.
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs,  item_vecs, model, u_s, i_s, 
                                                                       scaler, scalerUser, scalerItem, scaledata=scaledata)

print_pred_movies(sorted_ypu, sorted_user, sorted_items, movie_dict, maxcount = 10)

y_p,movie id,rating ave,title,genres
4.86762,64969,3.61765,Yes Man (2008),Comedy
4.86692,69122,3.63158,"Hangover, The (2009)",Comedy|Crime
4.86477,63131,3.625,Role Models (2008),Comedy
4.85853,60756,3.55357,Step Brothers (2008),Comedy
4.85785,68135,3.55,17 Again (2009),Comedy|Drama
4.85178,78209,3.55,Get Him to the Greek (2010),Comedy
4.85138,8622,3.48649,Fahrenheit 9/11 (2004),Documentary
4.8505,67087,3.52941,"I Love You, Man (2009)",Comedy
4.85043,69784,3.65,Brüno (Bruno) (2009),Comedy
4.84934,89864,3.63158,50/50 (2011),Comedy|Drama


Nếu bạn tạo một user ở trên thì cần lưu ý rằng network đã được training để dự đoán xếp hạng của user dựa trên vector, bao gồm một **bộ** xếp hạng thể loại của user.  Việc chỉ cung cấp xếp hạng tối đa cho một thể loại và xếp hạng tối thiểu cho các thể loại còn lại có thể không có ý nghĩa đối với network nếu không có user nào có nhóm xếp hạng tương tự.


#### Predictions cho user hiện tại.
Hãy xem predictions cho "user 36", một trong những user trong tập dữ liệu. Chúng ta có thể so sánh xếp hạng dự đoán với xếp hạng của model. Lưu ý rằng phim có nhiều thể loại sẽ hiển thị nhiều lần trong dữ liệu training. Ví dụ: 'The Time Machine' có ba thể loại: Adventure, Action, Sci-Fi


In [17]:
uid =  36 
# tạo thành một tập hợp người dùng vectors. Đây chính là vector, được biến đổi và lặp lại.
user_vecs, y_vecs = get_user_vecs(uid, scalerUser.inverse_transform(user_train), item_vecs, user_to_genre)

# chia tỷ lệ vectors và tạo predictions cho tất cả các phim. Trả về kết quả được sắp xếp theo đánh giá.
sorted_index, sorted_ypu, sorted_items, sorted_user = predict_uservec(user_vecs, item_vecs, model, u_s, i_s, scaler, 
                                                                      scalerUser, scalerItem, scaledata=scaledata)
sorted_y = y_vecs[sorted_index]

# in được sắp xếp predictions
print_existing_user(sorted_ypu, sorted_y.reshape(-1,1), sorted_user, sorted_items, item_features, ivs, uvs, movie_dict, maxcount = 10)

y_p,y,user,user genre ave,movie rating ave,title,genres
3.1,3.0,36,3.00,2.86,"Time Machine, The (2002)",Adventure
3.0,3.0,36,3.00,2.86,"Time Machine, The (2002)",Action
2.8,3.0,36,3.00,2.86,"Time Machine, The (2002)",Sci-Fi
2.3,1.0,36,1.00,4.00,"Beautiful Mind, A (2001)",Romance
2.2,1.0,36,1.50,4.00,"Beautiful Mind, A (2001)",Drama
1.6,1.5,36,1.75,3.52,Road to Perdition (2002),Crime
1.6,2.0,36,1.75,3.52,Gangs of New York (2002),Crime
1.5,1.5,36,1.50,3.52,Road to Perdition (2002),Drama
1.5,2.0,36,1.50,3.52,Gangs of New York (2002),Drama


#### Tìm các mục tương tự
neural network ở trên tạo ra hai feature vectors, một user feature vector $v_u$ và một movie feature vector, $v_m$. Đây là 32 mục vectors có giá trị khó diễn giải. Tuy nhiên, những item tương tự sẽ có vectors tương tự. Thông tin này có thể được sử dụng để đưa ra khuyến nghị. Ví dụ: nếu user đã đánh giá cao "Toy Story 3", người ta có thể đề xuất những bộ phim tương tự bằng cách chọn những bộ phim có movie feature vectors tương tự.

Thước đo tương tự là khoảng cách bình phương giữa hai vectors $ \mathbf{v_m^{(k)}}$ và $\mathbf{v_m^{(i)}}$ :
$$\left\Vert \mathbf{v_m^{(k)}} - \mathbf{v_m^{(i)}}  \right\Vert^2 = \sum_{l=1}^{n}(v_{m_l}^{(k)} - v_{m_l}^{(i)})^2\tag{1}$$


<a name="ex01"></a>
### Exercise 1

Viết hàm tính khoảng cách bình phương.


In [20]:
# GRADED_FUNCTION: sq_dist
# UNQ_C2
def sq_dist(a,b):
    """
    Returns the squared distance between two vectors
    Args:
      a (ndarray (n,)): vector with n features
      b (ndarray (n,)): vector with n features
    Returns:
      d (float) : distance
    """
    # ## START CODE HERE ###
    d = sum(np.square(a-b))
    # ## END CODE HERE ###
    return (d)

In [21]:
# Kiểm tra công khai
test_sq_dist(sq_dist)

All tests passed!


In [22]:
a1 = np.array([1.0, 2.0, 3.0]); b1 = np.array([1.0, 2.0, 3.0])
a2 = np.array([1.1, 2.1, 3.1]); b2 = np.array([1.0, 2.0, 3.0])
a3 = np.array([0, 1, 0]);       b3 = np.array([1, 0, 0])
print(f"squared distance between a1 and b1: {sq_dist(a1, b1)}")
print(f"squared distance between a2 and b2: {sq_dist(a2, b2)}")
print(f"squared distance between a3 and b3: {sq_dist(a3, b3)}")

squared distance between a1 and b1: 0.0
squared distance between a2 and b2: 0.030000000000000054
squared distance between a3 and b3: 2


<details>
  <summary><font size="3" color="darkgreen"><b>Click để biết gợi ý</b></font></summary>
    
  Mặc dù phép tính tổng thường là dấu hiệu nên sử dụng vòng lặp for, nhưng ở đây phép trừ có thể theo từng phần tử trong một câu lệnh. Hơn nữa, bạn có thể sử dụng np.square để tính bình phương, theo từng phần tử, kết quả của phép trừ. np.sum có thể được sử dụng để tính tổng các phần tử bình phương.
    
</details>


Matrix khoảng cách giữa các phim có thể được tính toán một lần khi model được training và sau đó được sử dụng lại cho các đề xuất mới mà không cần training lại. Bước đầu tiên, sau khi model được training, là lấy movie feature vector, $v_m$ cho mỗi phim. Để làm điều này, chúng ta sẽ sử dụng `item_NN` đã được training và xây dựng một model nhỏ để cho phép chúng ta chạy movie vectors thông qua nó để tạo $v_m$.


In [23]:
input_item_m = tf.keras.layers.Input(shape=(num_item_features))    # input layer
vm_m = item_NN(input_item_m)                                       # use the trained item_NN
vm_m = tf.linalg.l2_normalize(vm_m, axis=1)                        # incorporate normalization as was done in the original model
model_m = Model(input_item_m, vm_m)                                
model_m.summary()

Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None, 16)]         0                                            
__________________________________________________________________________________________________
sequential_1 (Sequential)       (None, 32)           41376       input_3[0][0]                    
__________________________________________________________________________________________________
tf_op_layer_l2_normalize_2/Squa [(None, 32)]         0           sequential_1[1][0]               
__________________________________________________________________________________________________
tf_op_layer_l2_normalize_2/Sum  [(None, 1)]          0           tf_op_layer_l2_normalize_2/Square
____________________________________________________________________________________________

Khi bạn có model phim, bạn có thể tạo một movie feature vectors bằng cách sử dụng model để dự đoán bằng cách sử dụng một bộ item/movie vectors là input. `item_vecs` là tập hợp tất cả của bộ movie vectors. Nhớ rằng cùng một bộ phim sẽ xuất hiện dưới dạng vector riêng biệt cho từng thể loại của nó. Nó phải được điều chỉnh tỷ lệ để sử dụng với model đã được training. Kết quả của prediction là feature vector gồm 32 mục cho mỗi phim.


In [24]:
scaled_item_vecs = scalerItem.transform(item_vecs)
vms = model_m.predict(scaled_item_vecs[:,i_s:])
print(f"size of all predicted movie feature vectors: {vms.shape}")

size of all predicted movie feature vectors: (1883, 32)


Bây giờ chúng ta hãy tính matrix của bình phương khoảng cách giữa mỗi movie feature vector và tất cả các movie feature vectors khác:
<figure>
    <left> <img src="./images/distmatrix.PNG"   style="width:400px;height:225px;" ></center>
</figure>


Sau đó, chúng ta có thể tìm phim gần nhất bằng cách tìm mức tối thiểu dọc theo mỗi hàng. Chúng ta sẽ sử dụng [numpy masked arrays](https://numpy.org/doc/1.21/user/tutorial-ma.html) để tránh chọn cùng một phim. Các giá trị bị che dọc theo đường chéo sẽ không được đưa vào tính toán.


In [25]:
count = 50
dim = len(vms)
dist = np.zeros((dim,dim))

for i in range(dim):
    for j in range(dim):
        dist[i,j] = sq_dist(vms[i, :], vms[j, :])
        
m_dist = ma.masked_array(dist, mask=np.identity(dist.shape[0]))  # mask the diagonal

disp = [["movie1", "genres", "movie2", "genres"]]
for i in range(count):
    min_idx = np.argmin(m_dist[i])
    movie1_id = int(item_vecs[i,0])
    movie2_id = int(item_vecs[min_idx,0])
    genre1,_  = get_item_genre(item_vecs[i,:], ivs, item_features)
    genre2,_  = get_item_genre(item_vecs[min_idx,:], ivs, item_features)

    disp.append( [movie_dict[movie1_id]['title'], genre1,
                  movie_dict[movie2_id]['title'], genre2]
               )
table = tabulate.tabulate(disp, tablefmt='html', headers="firstrow", floatfmt=[".1f", ".1f", ".0f", ".2f", ".2f"])
table

movie1,genres,movie2,genres
Save the Last Dance (2001),Drama,John Q (2002),Drama
Save the Last Dance (2001),Romance,Saving Silverman (Evil Woman) (2001),Romance
"Wedding Planner, The (2001)",Comedy,National Lampoon's Van Wilder (2002),Comedy
"Wedding Planner, The (2001)",Romance,Mr. Deeds (2002),Romance
Hannibal (2001),Horror,Final Destination 2 (2003),Horror
Hannibal (2001),Thriller,"Sum of All Fears, The (2002)",Thriller
Saving Silverman (Evil Woman) (2001),Comedy,Cats & Dogs (2001),Comedy
Saving Silverman (Evil Woman) (2001),Romance,Save the Last Dance (2001),Romance
Down to Earth (2001),Comedy,Joe Dirt (2001),Comedy
Down to Earth (2001),Fantasy,"Haunted Mansion, The (2003)",Fantasy


Kết quả cho thấy model sẽ gợi ý một bộ phim cùng thể loại.


<a name="4"></a>
## 4 - Congratulations! <img align="left" src="./images/film_award.png" style=" width:40px;">
Bạn đã hoàn thành recommender system dựa trên nội dung.    

Cấu trúc này là cơ sở của nhiều recommender systems thương mại. Nội dung user có thể được mở rộng đáng kể để kết hợp thêm thông tin về user nếu có.  Các mặt hàng không giới hạn ở phim. Điều này có thể được sử dụng để giới thiệu bất kỳ mặt hàng, sách, ô tô hoặc mặt hàng nào tương tự như một mặt hàng trong 'giỏ hàng' của bạn.
